# Recuperación privada opcional de los pesos originales V5

Este cuaderno está separado del diagnóstico estadístico NumPy/SciPy. **Está desactivado por defecto.** No entrena, no usa optimizador, no calibra, no modifica respaldos y no publica pesos ni predicciones.

Objetivo: recuperar exclusivamente el respaldo cifrado del commit `05c1a637f13356dd1df6b8c5aa8f4676dda61c12`, comprobar los cinco snapshots originales `input_only_mse` y producir **2930 predicciones retrospectivas**: 532 fit + 54 selección, en cada una de cinco semillas. Antes exige reproducir los 55 pronósticos de assessment guardados y la RMSE de selección por semilla; en total realiza 3205 puntos de inferencia. Este control de paridad es obligatorio.

**Estado de entrega:** 14 verificaciones de software, con fixtures y comprobación de archivos públicos/datos ya guardados; ninguna recuperación con clave real ni inferencia real nueva ha sido ejecutada por el asistente. El cuaderno no acredita adecuación, calibración ni R3.


In [ ]:
# Esta celda no instala dependencias ni solicita secretos.
RUN_RECOVERY = False
INSTALL_RUNTIME = False  # Sólo activar aquí si no existe ya un runtime CPU compatible.
PYTHON = None  # Ruta opcional a un Python con Torch2.8.0+cpu y NumPy2.3.5.
WORK = "/content/e3-v5-readonly-recovery"
CODE_DIR = "/content/e3-v5-readonly-code"


Para usarlo de forma voluntaria, activa `RUN_RECOVERY=True`. Si ya tienes un intérprete CPU compatible, escribe su ruta en `PYTHON`. La instalación aislada del lock original requiere activar también `INSTALL_RUNTIME=True`; el cuaderno estadístico principal no necesita esa instalación.

Sólo si todavía no existe el caso local se leen los secretos privados de Colab `GITHUB_TOKEN` (lectura) y **`E3_V5_BACKUP_KEY`** (la clave original usada en V5). Si no están en Colab, la celda los solicita mediante entrada oculta. No introduzcas claves en texto, URLs, código guardado, salidas ni en el chat. No sirve crear otra clave para un respaldo antiguo.

La fuente, el ZIP, el NPZ y el cifrado se fijan por commit, tamaño y SHA256. La consulta remota permite únicamente GET; no llama `setup`, `run`, `assess` ni `backup`. Usa el preprocesamiento original y sólo deserializa `training_raw`, `training_y_original` y `training_dates` del NPZ histórico; no abre lógicamente la reserva ni descarga mercado nuevo.


In [ ]:
# El código público está embebido con SHA; no depende de main ni de una URL futura.
import base64, hashlib, importlib.util, json
from pathlib import Path
PAYLOAD = {'frozen_v5_reanalysis.py': {'sha256': '1bc66b17b203d6dbfe2a45c50a245424c382d0c8ed1ae0fa050efcfd02d96a8b', 'base64': '...'}, 'colab_optional_recovery.py': {'sha256': 'adbf64dc645ec64d6c1bab77dfeb0ea7d87f6662e676013344ecb20966a4ac5d', 'base64': 'IiIiT3B0aW9uYWwgQ29sYWIgbGF1bmNoZXIuIENhbGxpbmcgZXhlY3V0ZSB3aXRoIGVuYWJsZWQ9RmFsc2UgcGVyZm9ybXMgbm8gSS9PLgoKTm8gc2V0dXAvcnVuL2Fzc2Vzcy9iYWNrdXAgaXMgY2FsbGVkLiBTZWNyZXRzIGFyZSByZWFkIG9ubHkgYnkgdGhlIGV4cGxpY2l0bHkKaW52b2tlZCBwcml2YXRlIHJlY292ZXJ5IGNlbGwgYW5kIHBhc3NlZCB2aWEgc3RkaW4sIG5ldmVyIGFyZ3YsIGVudmlyb25tZW50IG9yCm91dHB1dC4gT3JpZ2luYWwgc25hcHNob3RzIGFuZCBvdXRwdXQgQ1NWcyBzdGF5IGxvY2FsIGFuZCBwcml2YXRlLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwppbXBvcnQgaGFzaGxpYgppbXBvcnQgaW1wb3J0bGliLnV0aWwKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB0ZW1wZmlsZQppbXBvcnQgdXJsbGliLnJlcXVlc3QKaW1wb3J0IHZlbnYKClNPVVJDRV9DT01NSVQ9J2M0Y2E4NmYxNDMxYWZlYWE5NThmZjBiNTQ0YjQ1M2FiMTdhNGQ3YjgnCk1BTklGRVNUX1NIQT0nNDcwNmYyMGEwMDUwOGU0OTJhNTRlMjU4NzNiNDg4NWNlYWQzZmE3NjYwZDQzYmRiMThjNWFlZGE0ZmNlMjUwOScKQk9PVFNUUkFQX1NIQT0nM2E2YmNlOTllMTQ5ZTNhMWU2Y2ZiYjA3YjhlMjNmYjY0ODJkOWJiODc2NTk3ZDRmOGQ1ZGYxOThjZGUwNzk0MScKQkFTRT0naHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL2phc3Nhbi1iaXQvRGVlcExlYXJuaW5nXzIwMjYzMC8nClNPVVJDRV9CQVNFPUJBU0UrU09VUkNFX0NPTU1JVCsnL2V4cGVyaW1lbnRzL2UzX3Nlb2lfdjVfcGlsb3RfMjAyNjExMjEvJwpEQVRBX1VSTD1CQVNFKyc4ZjY1ZWFjMzRhYWY0OTI5MzkyNGFlODMyOGMyMzVjZWRiZGQyODY2L2Jvb2tfZTIvX3N0YXRpYy9lMl9zY2llbnRpZmljX2Nsb3N1cmVfMjAyNjExMTYvZGF0YV9hdWRpdC9jdXJyZW50X2NvaG9ydF9hcnJheXMubnB6JwpSRVNVTFRfVVJMPUJBU0UrJzA1YzFhNjM3ZjEzMzU2ZGQxZGY2YjhjNWFhOGY0Njc2ZGRhNjFjMTIvcnVucy9lM19zZW9pX3Y1L3J1bi0wMS9wdWJsaWNfcmVzdWx0cy56aXAnCkRBVEFfU0hBPScxODZkNDczZmRmZGY1Y2Q0NDEzZWIxMjAwZGZjYzhmZWE2NWJhOWY2ZTc1MTliNzIyZWI5YjFhZTlmNTZhNjgzJwpSRVNVTFRfU0hBPSdiZWFiZWRlYjYzZmYxMDMyNGIzMTBiYThiODYxMDU1MDVjM2JlNzgzMjE2ZTM4NzcwYTA5MmE3YmU0MzhhMTdjJwoKCmRlZiBkb3dubG9hZF92ZXJpZmllZCh1cmwscGF0aCxleHBlY3RlZCxzaXplPU5vbmUsbGltaXQ9MTA0ODU3Nik6CiAgICBwYXRoPVBhdGgocGF0aCkKICAgIGlmIHBhdGguaXNfc3ltbGluaygpOnJhaXNlIFJ1bnRpbWVFcnJvcignU3ltbGluayByZWZ1c2VkJykKICAgIGlmIHBhdGguZXhpc3RzKCk6cmF3PXBhdGgucmVhZF9ieXRlcygpCiAgICBlbHNlOgogICAgICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3Blbih1cmwsdGltZW91dD02MCkgYXMgc3RyZWFtOnJhdz1zdHJlYW0ucmVhZChsaW1pdCsxKQogICAgaWYgbGVuKHJhdyk+bGltaXQgb3IgKHNpemUgaXMgbm90IE5vbmUgYW5kIGxlbihyYXcpIT1zaXplKSBvciBoYXNobGliLnNoYTI1NihyYXcpLmhleGRpZ2VzdCgpIT1leHBlY3RlZDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ1Bpbm5lZCBwdWJsaWMgZG93bmxvYWQgZGlmZmVyczsgcHJlc2VydmUgY2FjaGUgYW5kIHN0b3AnKQogICAgaWYgbm90IHBhdGguZXhpc3RzKCk6CiAgICAgICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICAgICAgZmQsdG1wPXRlbXBmaWxlLm1rc3RlbXAocHJlZml4PScudmVyaWZpZWQtJyxkaXI9cGF0aC5wYXJlbnQpCiAgICAgICAgdHJ5OgogICAgICAgICAgICB3aXRoIG9zLmZkb3BlbihmZCwnd2InKSBhcyBzdHJlYW06c3RyZWFtLndyaXRlKHJhdykKICAgICAgICAgICAgb3MuY2htb2QodG1wLDBvNjAwKTtvcy5yZXBsYWNlKHRtcCxwYXRoKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKHRtcCk6b3MudW5saW5rKHRtcCkKICAgIHJldHVybiBwYXRoCgoKZGVmIGNwdV9lbnYoKToKICAgIHJlc3VsdD17azp2IGZvciBrLHYgaW4gb3MuZW52aXJvbi5pdGVtcygpIGlmIGsgbm90IGluIHsnUFlUSE9OUEFUSCcsJ1BZVEhPTkhPTUUnfSBhbmQKICAgICAgICBub3QgYW55KHdvcmQgaW4gay51cHBlcigpIGZvciB3b3JkIGluICgnVE9LRU4nLCdTRUNSRVQnLCdCQUNLVVBfS0VZJywnUEFTU1dPUkQnLCdDUkVERU5USUFMJykpfQogICAgcmVzdWx0LnVwZGF0ZShDVURBX1ZJU0lCTEVfREVWSUNFUz0nJyxPTVBfTlVNX1RIUkVBRFM9JzInLE1LTF9OVU1fVEhSRUFEUz0nMicsCiAgICAgICAgT1BFTkJMQVNfTlVNX1RIUkVBRFM9JzEnLFBZVEhPTkhBU0hTRUVEPScwJyxNUExCQUNLRU5EPSdBZ2cnKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBjb21wYXRpYmxlKHB5dGhvbik6CiAgICBwcm9iZT0naW1wb3J0IG51bXB5LHRvcmNoLGNyeXB0b2dyYXBoeTthc3NlcnQgbnVtcHkuX192ZXJzaW9uX189PSIyLjMuNSI7YXNzZXJ0IHRvcmNoLl9fdmVyc2lvbl9fPT0iMi44LjArY3B1IicKICAgIHRyeTpyZXR1cm4gc3VicHJvY2Vzcy5ydW4oW3N0cihweXRob24pLCctYycscHJvYmVdLGVudj1jcHVfZW52KCksY2FwdHVyZV9vdXRwdXQ9VHJ1ZSx0aW1lb3V0PTYwKS5yZXR1cm5jb2RlPT0wCiAgICBleGNlcHQgKE9TRXJyb3Isc3VicHJvY2Vzcy5UaW1lb3V0RXhwaXJlZCk6cmV0dXJuIEZhbHNlCgoKZGVmIHNlbGVjdF9weXRob24ocHJlZmVycmVkLHNvdXJjZSx3b3JrLGluc3RhbGxfcnVudGltZSk6CiAgICBjaG9pY2VzPVtwcmVmZXJyZWRdIGlmIHByZWZlcnJlZCBlbHNlIFtdCiAgICBjaG9pY2VzLmV4dGVuZChbd29yay8ncnVudGltZScvICdiaW4vcHl0aG9uJyxzeXMuZXhlY3V0YWJsZV0pCiAgICBmb3IgY2hvaWNlIGluIGNob2ljZXM6CiAgICAgICAgaWYgY2hvaWNlIGFuZCBjb21wYXRpYmxlKGNob2ljZSk6cmV0dXJuIFBhdGgoY2hvaWNlKQogICAgaWYgbm90IGluc3RhbGxfcnVudGltZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ05vIGNvbXBhdGlibGUgQ1BVIHJ1bnRpbWUuIFNldCBJTlNUQUxMX1JVTlRJTUU9VHJ1ZSBvbmx5IGluIHRoaXMgb3B0aW9uYWwgbm90ZWJvb2ssIG9yIHNldCBQWVRIT04gdG8gYW4gZXhpc3RpbmcgVG9yY2gyLjguMCtjcHUgLyBOdW1QeTIuMy41IGludGVycHJldGVyLiBObyBzZWNyZXRzIGhhdmUgYmVlbiByZXF1ZXN0ZWQuJykKICAgIGRpcmVjdG9yeT13b3JrLydydW50aW1lJwogICAgaWYgZGlyZWN0b3J5LmV4aXN0cygpOnJhaXNlIFJ1bnRpbWVFcnJvcignUHJlc2VydmUgaW5jb21wYXRpYmxlIHJ1bnRpbWUgYW5kIGNob29zZSBhIGZyZXNoIFdPUksgZGlyZWN0b3J5JykKICAgIHZlbnYuRW52QnVpbGRlcih3aXRoX3BpcD1UcnVlKS5jcmVhdGUoZGlyZWN0b3J5KQogICAgcHl0aG9uPWRpcmVjdG9yeS8nYmluL3B5dGhvbicKICAgICMgT25seSB0aGUgdmVyaWZpZWQgb3JpZ2luYWwgbG9jaywgaW4gdGhpcyBpc29sYXRlZCBvcHRpb25hbCBlbnZpcm9ubWVudC4KICAgIHN1YnByb2Nlc3MucnVuKFtzdHIocHl0aG9uKSwnLW0nLCdwaXAnLCdpbnN0YWxsJywnLS1kaXNhYmxlLXBpcC12ZXJzaW9uLWNoZWNrJywnLXInLHN0cihzb3VyY2UvJ3JlcXVpcmVtZW50cy1jcHUubG9jay50eHQnKV0sCiAgICAgICAgZW52PWNwdV9lbnYoKSxjaGVjaz1UcnVlKQogICAgaWYgbm90IGNvbXBhdGlibGUocHl0aG9uKTpyYWlzZSBSdW50aW1lRXJyb3IoJ09wdGlvbmFsIENQVSBydW50aW1lIHZlcmlmaWNhdGlvbiBmYWlsZWQnKQogICAgcmV0dXJuIHB5dGhvbgoKCmRlZiBwcml2YXRlX3NlY3JldChuYW1lLHByb21wdCk6CiAgICB0cnk6CiAgICAgICAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IHVzZXJkYXRhCiAgICAgICAgdmFsdWU9dXNlcmRhdGEuZ2V0KG5hbWUpCiAgICAgICAgaWYgdmFsdWU6cmV0dXJuIHZhbHVlCiAgICBleGNlcHQgRXhjZXB0aW9uOnBhc3MKICAgIGltcG9ydCBnZXRwYXNzCiAgICByZXR1cm4gZ2V0cGFzcy5nZXRwYXNzKHByb21wdCkKCgpkZWYgZXhlY3V0ZSgqLGVuYWJsZWQ9RmFsc2UsaW5zdGFsbF9ydW50aW1lPUZhbHNlLHB5dGhvbj1Ob25lLHdvcms9Jy9jb250ZW50L2UzLXY1LXJlYWRvbmx5LXJlY292ZXJ5JywKICAgICAgICAgICAgaGVscGVyPU5vbmUsaGVscGVyX3NoYTI1Nj1Ob25lKToKICAgIGlmIG5vdCBlbmFibGVkOgogICAgICAgIHJldHVybiB7J3N0YXR1cyc6J0RJU0FCTEVEX05PX0RPV05MT0FEX05PX0lOU1RBTExfTk9fU0VDUkVUX05PX0lORkVSRU5DRScsJ3RyYWluaW5nX3N0ZXBzJzowLCdyZW1vdGVfd3JpdGVzJzowfQogICAgaGVscGVyPVBhdGgoaGVscGVyKS5yZXNvbHZlKCkKICAgIGlmIGhhc2hsaWIuc2hhMjU2KGhlbHBlci5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpIT1oZWxwZXJfc2hhMjU2OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignUmVjb3ZlcnkgaGVscGVyIGRpZmZlcnMgZnJvbSBub3RlYm9vayBzb3VyY2UgcGluJykKICAgIHdvcms9UGF0aChvcy5wYXRoLmFic3BhdGgod29yaykpCiAgICBpZiBhbnkocC5pc19zeW1saW5rKCkgZm9yIHAgaW4gKHdvcmssKndvcmsucGFyZW50cykpOnJhaXNlIFJ1bnRpbWVFcnJvcignU3ltbGluayB3b3Jrc3BhY2UgcmVmdXNlZCcpCiAgICB3b3JrLm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKTtvcy5jaG1vZCh3b3JrLDBvNzAwKQogICAgaWYgd29yay5pc19yZWxhdGl2ZV90byhoZWxwZXIucGFyZW50KToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ1ByaXZhdGUgd29yayBtdXN0IGJlIG91dHNpZGUgdGhlIHB1YmxpYyBjb2RlIHBhY2tldCcpCiAgICBvbGRfdW1hc2s9b3MudW1hc2soMG8wNzcpCiAgICB0cnk6CiAgICAgICAgb3V0cHV0PXdvcmsvJ2ZpdF9zZWxlY3Rpb25fUFJJVkFURScKICAgICAgICBpZiBvdXRwdXQuZXhpc3RzKCk6CiAgICAgICAgICAgIHByb2Nlc3M9c3VicHJvY2Vzcy5ydW4oW3N5cy5leGVjdXRhYmxlLHN0cihoZWxwZXIpLCctLW91dHB1dC1kaXInLHN0cihvdXRwdXQpLCctLXZlcmlmeS1vbmx5J10sCiAgICAgICAgICAgICAgICBpbnB1dD0ne30nLHRleHQ9VHJ1ZSxzdGRvdXQ9c3VicHJvY2Vzcy5QSVBFLHN0ZGVycj1zdWJwcm9jZXNzLlBJUEUsZW52PWNwdV9lbnYoKSx0aW1lb3V0PTYwKQogICAgICAgICAgICB0cnk6cmV0dXJuIGpzb24ubG9hZHMocHJvY2Vzcy5zdGRvdXQuc3RyaXAoKS5zcGxpdGxpbmVzKClbLTFdKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOnJldHVybiB7J3N0YXR1cyc6J1NUT1BQRUQnLCdkaWFnbm9zdGljJzonRXhpc3Rpbmcgb3V0cHV0IHZlcmlmaWNhdGlvbiBmYWlsZWQ7IHByaXZhdGUgZmlsZXMgcHJlc2VydmVkJ30KICAgICAgICBib290c3RyYXA9ZG93bmxvYWRfdmVyaWZpZWQoU09VUkNFX0JBU0UrJ2Jvb3RzdHJhcF9jb2xhYi5weScsd29yay8nY2FjaGUvYm9vdHN0cmFwX2NvbGFiLnB5JyxCT09UU1RSQVBfU0hBKQogICAgICAgIHNwZWM9aW1wb3J0bGliLnV0aWwuc3BlY19mcm9tX2ZpbGVfbG9jYXRpb24oJ29yaWdpbmFsX3Y1X2Jvb3RzdHJhcF9yZWFkb25seV9hY3F1aXNpdGlvbicsYm9vdHN0cmFwKQogICAgICAgIG1vZHVsZT1pbXBvcnRsaWIudXRpbC5tb2R1bGVfZnJvbV9zcGVjKHNwZWMpO3NwZWMubG9hZGVyLmV4ZWNfbW9kdWxlKG1vZHVsZSkKICAgICAgICAjIGFjcXVpcmVfc291cmNlIG9ubHk6IG5ldmVyIHNldHVwKCksIHJ1bm5lci5ydW4oKSBvciBHaXRodWJTdG9yZS5iYWNrdXAoKS4KICAgICAgICBzb3VyY2U9bW9kdWxlLmFjcXVpcmVfc291cmNlKFNPVVJDRV9DT01NSVQsTUFOSUZFU1RfU0hBLHdvcmsvJ2NhY2hlJykKICAgICAgICBpbnRlcnByZXRlcj1zZWxlY3RfcHl0aG9uKHB5dGhvbixzb3VyY2Usd29yayxpbnN0YWxsX3J1bnRpbWUpCiAgICAgICAgZGF0YT1kb3dubG9hZF92ZXJpZmllZChEQVRBX1VSTCx3b3JrLydjYWNoZS9jb2hvcnQubnB6JyxEQVRBX1NIQSw0NjAwOTApCiAgICAgICAgcHVibGljPWRvd25sb2FkX3ZlcmlmaWVkKFJFU1VMVF9VUkwsd29yay8nY2FjaGUvcHVibGljX3Jlc3VsdHMuemlwJyxSRVNVTFRfU0hBLDEyNDA2NykKICAgICAgICBjYXNlPXdvcmsvJ29yaWdpbmFsX3ByaXZhdGVfY2FzZScKICAgICAgICBhcmdzPVtzdHIoaW50ZXJwcmV0ZXIpLHN0cihoZWxwZXIpLCctLW91dHB1dC1kaXInLHN0cihvdXRwdXQpXQogICAgICAgIGNyZWRlbnRpYWxzPXt9CiAgICAgICAgaWYgb3V0cHV0LmV4aXN0cygpOmFyZ3MrPVsnLS12ZXJpZnktb25seSddCiAgICAgICAgZWxzZToKICAgICAgICAgICAgYXJncys9WyctLXNvdXJjZS1yb290JyxzdHIoc291cmNlKSwnLS1kYXRhLWZpbGUnLHN0cihkYXRhKSwnLS1wdWJsaWMtemlwJyxzdHIocHVibGljKSwnLS1jYXNlLWRpcicsc3RyKGNhc2UpXQogICAgICAgICAgICBpZiBub3QgY2FzZS5leGlzdHMoKToKICAgICAgICAgICAgICAgIGNyZWRlbnRpYWxzWyd0b2tlbiddPXByaXZhdGVfc2VjcmV0KCdHSVRIVUJfVE9LRU4nLCdUb2tlbiBHaXRIdWIgZGUgbGVjdHVyYSAob2N1bHRvOyBubyBzZSBwdWJsaWNhcsOhKTogJykKICAgICAgICAgICAgICAgIGNyZWRlbnRpYWxzWydiYWNrdXBfc2VjcmV0J109cHJpdmF0ZV9zZWNyZXQoJ0UzX1Y1X0JBQ0tVUF9LRVknLCdDbGF2ZSBPUklHSU5BTCBkZSByZXNwYWxkbyBWNSAob2N1bHRhOyBubyBjcmVhciB1bmEgbnVldmEpOiAnKQogICAgICAgIHRyeToKICAgICAgICAgICAgcHJvY2Vzcz1zdWJwcm9jZXNzLnJ1bihhcmdzLGlucHV0PWpzb24uZHVtcHMoY3JlZGVudGlhbHMpLHRleHQ9VHJ1ZSxzdGRvdXQ9c3VicHJvY2Vzcy5QSVBFLHN0ZGVycj1zdWJwcm9jZXNzLlBJUEUsCiAgICAgICAgICAgICAgICBlbnY9Y3B1X2VudigpLHRpbWVvdXQ9OTAwKQogICAgICAgIGZpbmFsbHk6Y3JlZGVudGlhbHMuY2xlYXIoKQogICAgICAgICMgVGhlIGNoaWxkIHNhbml0aXplcyBleGNlcHRpb25zLiBEbyBub3QgZGlzcGxheSBzdGRlcnIsIGVudmlyb25tZW50IG9yIGNyZWRlbnRpYWxzLgogICAgICAgIHRyeTpyZXN1bHQ9anNvbi5sb2Fkcyhwcm9jZXNzLnN0ZG91dC5zdHJpcCgpLnNwbGl0bGluZXMoKVstMV0pCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjpyZXN1bHQ9eydzdGF0dXMnOidTVE9QUEVEJywnZGlhZ25vc3RpYyc6J1ByaXZhdGUgc3VicHJvY2VzcyBzdG9wcGVkLiBPcmlnaW5hbCBmaWxlcyBwcmVzZXJ2ZWQ7IG5vIHJhdyBzdGRlcnIgZGlzcGxheWVkLid9CiAgICAgICAgaWYgcHJvY2Vzcy5yZXR1cm5jb2RlOnJlc3VsdFsnZXhpdF9jb2RlJ109cHJvY2Vzcy5yZXR1cm5jb2RlCiAgICAgICAgcmV0dXJuIHJlc3VsdAogICAgZmluYWxseTpvcy51bWFzayhvbGRfdW1hc2spCg=='}}
code_dir = Path(CODE_DIR)
code_dir.mkdir(parents=True, exist_ok=True)
for name, item in PAYLOAD.items():
    raw = base64.b64decode(item["base64"], validate=True)
    assert hashlib.sha256(raw).hexdigest() == item["sha256"]
    target = code_dir / name
    assert not target.is_symlink()
    if target.exists():
        assert target.read_bytes() == raw, "Código local distinto: conservarlo y usar otro CODE_DIR"
    else:
        target.write_bytes(raw)
spec = importlib.util.spec_from_file_location("optional_private_v5_launcher", code_dir / "colab_optional_recovery.py")
launcher = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launcher)
result = launcher.execute(enabled=RUN_RECOVERY, install_runtime=INSTALL_RUNTIME, python=PYTHON,
    work=WORK, helper=code_dir / "frozen_v5_reanalysis.py",
    helper_sha256="1bc66b17b203d6dbfe2a45c50a245424c382d0c8ed1ae0fa050efcfd02d96a8b")
print(json.dumps(result, indent=2))


Los CSV quedan en `WORK/fit_selection_PRIVATE`, con manifiesto SHA256 y recibo de ejecución. El caso restaurado conserva sus metadatos y pesos; la salida se escribe en otra carpeta privada. Una segunda ejecución verifica la salida existente, sin nueva inferencia, dependencias ni secretos. Una salida incompleta o distinta se conserva y bloquea la reutilización; no se sobreescribe.

El recibo declara las versiones originales y las del intérprete real. El entrenamiento usó Python3.13.16; otra versión de inferencia se permite sólo como análisis independiente con las verificaciones de paridad, sin relajar el guard de reanudación original. Una discrepancia bloquea la exportación; **no ajustes tolerancias tras observar el resultado** (`atol=1e-8`, `rtol=1e-5`).

Conservar los CSV permite estudiar posteriormente la adecuación de fit/selección, con sus límites retrospectivos. No convierte calendarios expuestos en confirmación, no construye un margen prospectivo ni genera intervalos predictivos calibrados. El soporte de átomos del modelo no constituye por sí solo un intervalo válido.

Los archivos privados viven en la VM. Si deseas conservarlos, descárgalos desde el panel de archivos a un dispositivo privado. Este cuaderno no los envía a GitHub ni al depósito público. No compartas las salidas privadas sin revisar derechos y alcance. Asistencia de IA declarada; las fuentes y los datos conservan sus licencias y términos originales.
